# dehydration of the raw datasets

builds the publishable parquet files from the private raw data. this notebook only runs internally, the raw files never leave the server. three protections are applied.

1. all free text and profile fields are removed
2. all url and domain columns are removed, otherwise the per domain newsguard score table could be reconstructed from the published rows. the urls survive only as salted hashes, which keeps the lead time pair matching reproducible without revealing a single link
3. every platform identifier is replaced by a salted hash, which blocks rehydration through the x api but keeps retweet chains and per user groupings intact

In [1]:
import hashlib
import re
import secrets
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq

SRC_DIR = Path("/home/mangermaier/cs2/twitter/user")
OUT_DIR = SRC_DIR / "dehydrated_for_publication"
SALT_FILE = SRC_DIR / "dehydration_salt_PRIVATE.txt"

SOURCES = {
    "sensor": SRC_DIR / "result_sensor_with_users_unraveled_v5.parquet",
    "random": SRC_DIR / "result_random_with_users_unraveled_v5.parquet",
}

## the salt

the identifiers are hashed with a secret salt. without it nobody can brute force the hashes from known tweet ids, with it we can reproduce the mapping internally at any time. the salt lives next to the private raw data and is never published or committed.

In [2]:
if SALT_FILE.exists():
    salt = SALT_FILE.read_text().strip().encode()
    print("using existing salt")
else:
    s = secrets.token_hex(32)
    SALT_FILE.write_text(s + "\n")
    SALT_FILE.chmod(0o600)
    salt = s.encode()
    print("generated new salt")

using existing salt


## what stays and what goes

four id columns get hashed. the url list gets hashed element by element, two tweets sharing a link share the hash, but nobody can go from hash to link. seventeen columns are carried over unchanged, they are exactly what the paper analyses need, timestamps, engagement counts, the newsguard values without their domains, and the user level counts for the friendship paradox validation. everything else is dropped, that covers tweet text, user profiles, mentions, the clear urls and domains, unused api metadata and unused model scores.

In [3]:
HASH_COLS = ["TWEET_id", "USER_id", "referenced_tweet_id", "conversation_id"]

# hashed element by element, published as urls_hashed
HASH_LIST_COLS = ["urls_expanded"]

KEEP_COLS = [
    "referenced_tweet_type",
    "TWEET_created_at",
    "lang",
    "TWEET_like_count",
    "retweet_count",
    "quote_count",
    "bookmark_count",
    "reply_count",
    "impression_count",
    "newsguard_scores_expanded",
    "newsguard_orientation",
    "USER_followers_count",
    "USER_following_count",
    "USER_tweet_count",
    "USER_listed_count",
    "USER_verified",
    "USER_created_at",
]

# published under a name without the internal suffix
RENAME = {"newsguard_scores_expanded": "newsguard_scores"}

## hashing

sha-256 over salt plus id, truncated to 16 hex characters. the cache speeds up the many repeats of the same user id, one user contributes thousands of rows.

In [4]:
cache = {}

def hash_id(value):
    if value is None or value == "":
        return None
    h = cache.get(value)
    if h is None:
        h = hashlib.sha256(salt + value.encode()).hexdigest()[:16]
        cache[value] = h
    return h

# urls are mostly unique, caching them would eat memory for nothing
def hash_raw(value):
    if value is None or value == "":
        return None
    return hashlib.sha256(salt + value.encode()).hexdigest()[:16]

## streaming the two files

200k rows per batch, hash the id columns, carry the kept columns over, write with snappy compression. the same cache spans both files, so a user appearing in both gets the same hash.

In [5]:
for name, src in SOURCES.items():
    out_path = OUT_DIR / f"{name}_dehydrated.parquet"
    pf = pq.ParquetFile(src)
    available = set(pf.schema_arrow.names)
    hash_cols = [c for c in HASH_COLS if c in available]
    list_cols = [c for c in HASH_LIST_COLS if c in available]
    keep_cols = [c for c in KEEP_COLS if c in available]
    writer = None
    done = 0
    for batch in pf.iter_batches(batch_size=200_000,
                                 columns=hash_cols + list_cols + keep_cols):
        d = batch.to_pydict()
        arrays, names = [], []
        for c in hash_cols:
            names.append(c + "_hashed")
            arrays.append(pa.array([hash_id(v) for v in d[c]],
                                   type=pa.string()))
        for c in list_cols:
            names.append("urls_hashed")
            arrays.append(pa.array(
                [None if lst is None else [hash_raw(u) for u in lst]
                 for lst in d[c]], type=pa.list_(pa.string())))
        for c in keep_cols:
            names.append(RENAME.get(c, c))
            arrays.append(batch.column(batch.schema.get_field_index(c)))
        table = pa.table(dict(zip(names, arrays)))
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema,
                                      compression="snappy")
        writer.write_table(table, row_group_size=1_000_000)
        done += len(batch)
    writer.close()
    print(f"{name}: {done:,} rows -> {out_path.name}, "
          f"{out_path.stat().st_size / 1e9:.2f} gb")

sensor: 30,964,074 rows -> sensor_dehydrated.parquet, 2.10 gb


random: 27,395,048 rows -> random_dehydrated.parquet, 1.84 gb


## check the result

no forbidden column present, row counts match the sources, hashes well formed, newsguard values present without any domain next to them.

In [6]:
forbidden = {"text", "urls", "urls_expanded", "domain_from_url",
             "domain_from_urls_expanded", "USER_username", "USER_name",
             "USER_description", "USER_location", "mentions",
             "TWEET_id", "USER_id", "referenced_tweet_id", "conversation_id"}
hex16 = re.compile(r"^[0-9a-f]{16}$")
for name, src in SOURCES.items():
    out = pq.ParquetFile(OUT_DIR / f"{name}_dehydrated.parquet")
    cols = set(out.schema_arrow.names)
    assert not (cols & forbidden), cols & forbidden
    assert out.metadata.num_rows == pq.ParquetFile(src).metadata.num_rows
    sample = next(out.iter_batches(batch_size=50_000)).to_pydict()
    assert all(hex16.match(v) for v in sample["USER_id_hashed"] if v)
    assert all(hex16.match(v) for v in sample["TWEET_id_hashed"] if v)
    assert all(hex16.match(u) for lst in sample["urls_hashed"] if lst
               for u in lst if u)
    ng = [s for s in sample["newsguard_scores"]
          if s is not None and any(x is not None for x in s)]
    print(f"{name}: {out.metadata.num_rows:,} rows, {len(cols)} columns, "
          f"{len(ng):,} of 50,000 sampled rows carry newsguard values")
print("all checks passed")

sensor: 30,964,074 rows, 22 columns, 2,384 of 50,000 sampled rows carry newsguard values


random: 27,395,048 rows, 22 columns, 48 of 50,000 sampled rows carry newsguard values
all checks passed
